# 🚀 Full LLM Fine-Tuning on 100% Dataset (AMD MI300X)

Fine-tuning pipeline for **Qwen2.5-Coder-32B-Instruct** on the Java Vulnerability Dataset.
Optimised for **AMD MI300X (192GB VRAM)** — TRL 1.6.x compatible.

**Key fixes vs the 16K iteration:**
- ✅ `SYSTEM_PROMPT` defined inside Cell 5 (self-contained, no cross-cell dependency)
- ✅ `packing=False` → prevents cross-contamination and 60K token overflow
- ✅ `max_seq_length=2048` in SFTConfig (correct location for TRL 1.6)
- ✅ `modules_to_save=[embed_tokens, lm_head]` → preserves Java syntax precision
- ✅ `learning_rate=1e-4` → safer for 32B, avoids catastrophic forgetting
- ✅ `warmup_steps=100` (replaces deprecated warmup_ratio)
- ✅ Improved truncation filter + fallback formatter for no-code-block examples

### Dependencies Installation
**What this cell does:** Installs the required libraries for fine-tuning.

**Important Libraries:**
*   `transformers`, `peft`, `trl`: Hugging Face libraries for loading models, applying LoRA (PEFT), and the SFTTrainer (TRL).
*   `bitsandbytes`: For quantization (if used).

In [ ]:
# ─── Cell 1: Install Dependencies ─────────────────────────────────────────────
!pip install -U transformers peft trl datasets accelerate
!pip install bitsandbytes

### Imports and Setup
**What this cell does:** Imports the required Python modules and prints the environment versions to verify GPU availability.

In [ ]:
# ─── Cell 2: Imports ──────────────────────────────────────────────────────────
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

import trl
print(f"TRL version  : {trl.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA/ROCm GPUs: {torch.cuda.device_count()}")

### Global Configuration
**What this cell does:** Sets up high-level toggles for the training run.

**Important Parameters:**
*   **`MODEL_SIZE = "32B"`**: Selects the 32-Billion parameter Qwen model.
*   **`USE_4BIT = False`**: Disables 4-bit quantization, allowing native bfloat16 training on the 192GB MI300X for maximum performance.

In [ ]:
# ─── Cell 3: Configuration ────────────────────────────────────────────────────
MODEL_SIZE = "32B"   # Options: "32B" or "7B"
USE_4BIT   = False   # False = native bfloat16 LoRA (recommended for ROCm MI300X)
EPOCHS     = 1       # 1 epoch over the full dataset

print(f"Config: Qwen2.5-Coder-{MODEL_SIZE}-Instruct | 4-bit={USE_4BIT} | Epochs={EPOCHS}")

### Data Preparation and Cleaning
**What this cell does:** Loads the JSONL datasets and filters out corrupted or incomplete data.

**Important Functions:**
*   **`is_complete_example`**: Validates each row. It drops completions shorter than 30 characters and checks for unclosed ```java blocks. This ensures the model only trains on 100% complete syntax, completely preventing it from learning to generate truncated code.

In [ ]:
# ─── Cell 4: Load and Clean the Dataset ───────────────────────────────────────
data_files = {
    "train":      "train.jsonl",
    "validation": "val.jsonl",
    "test":       "test.jsonl"
}
raw_dataset = load_dataset("json", data_files=data_files)
print(f"Original Train size: {len(raw_dataset['train'])}")

def is_complete_example(example):
    """Filter out truncated or empty training examples."""
    completion = (example.get('completion', '') or example.get('output', '') or '').strip()
    input_code = example.get('input', '').strip()

    # Drop if completion is too short to be a real fix
    if len(completion) < 30:
        return False

    # Safe examples (input == output) — always keep
    if input_code and input_code == completion:
        return True

    # Vulnerable examples: check code block is closed (not truncated mid-way)
    if "```java" in completion:
        parts = completion.split("```java")
        if len(parts) >= 2 and "```" not in parts[1]:
            return False  # opened but never closed — truncated

    return True

dataset = raw_dataset.filter(is_complete_example)
print(f"Cleaned Train size : {len(dataset['train'])} (removed truncated/empty examples)")
print(f"Validation size    : {len(dataset['validation'])}")

### Tokenizer and Chat Template Formatting
**What this cell does:** Wraps the raw data into a structured conversational format expected by the model.

**Important Parameters & Functions:**
*   **`SYSTEM_PROMPT`**: Injects a strict persona (`You are an expert Java security auditor...`) to guide the model's baseline behavior.
*   **`format_chat_template`**: Converts the instruction, input, and output into a `System -> User -> Assistant` format. For vulnerable code, it strictly enforces a markdown structure (`### 🛡️ Vulnerability Analysis`, etc.). By training on this precise structure, the model learns to output highly predictable, parseable reports.

In [ ]:
# ─── Cell 5: Tokenizer & Chat Template ────────────────────────────────────────
#
# SYSTEM_PROMPT is defined HERE (not in Cell 3) so this cell is fully
# self-contained. Running cells out of order will no longer cause NameError.

SYSTEM_PROMPT = """You are an expert Java security auditor. Analyze the provided code.
If the code is secure, output:
\"This Java code is completely secure and contains no vulnerabilities. No changes are required.\"

If the code is vulnerable, output your analysis in this exact format:
### \U0001f6e1\ufe0f Vulnerability Analysis
*   **Status**: VULNERABLE
*   **Type**: [Vulnerability Type]
*   **Severity**: HIGH

### \U0001f4dd Explanation
[Provide a brief explanation of the vulnerability]

### \U0001f6e0\ufe0f Fixed Code
```java
[Fixed complete Java code]
```"""

model_id = "Qwen/Qwen2.5-Coder-32B-Instruct" if MODEL_SIZE == "32B" else "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    if "messages" in example:
        example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
        return example

    instruction        = example.get('prompt', '') or example.get('instruction', '')
    input_code         = example.get('input', '')
    completion         = example.get('completion', '') or example.get('output', '')
    vulnerability_type = example.get('vulnerability_type', 'Vulnerable')

    full_prompt = f"{instruction}\n\n{input_code}" if input_code else instruction

    # ── Build the assistant reply ──────────────────────────────────────────────
    if vulnerability_type == "Safe" or (input_code and input_code.strip() == completion.strip()):
        # Safe code: exact string that inference notebooks check for
        assistant_reply = "This Java code is completely secure and contains no vulnerabilities. No changes are required."

    elif "```java" in completion:
        # Vulnerable code WITH a java block — restructure into standard template
        parts       = completion.split("```java")
        explanation = parts[0].strip()
        code_block  = parts[1].split("```")[0].strip()
        assistant_reply = (
            f"### \U0001f6e1\ufe0f Vulnerability Analysis\n"
            f"*   **Status**: VULNERABLE\n"
            f"*   **Type**: {vulnerability_type}\n"
            f"*   **Severity**: HIGH\n\n"
            f"### \U0001f4dd Explanation\n"
            f"{explanation}\n\n"
            f"### \U0001f6e0\ufe0f Fixed Code\n"
            f"```java\n{code_block}\n```"
        )
    else:
        # Fallback: no code block — wrap explanation in standard header
        assistant_reply = (
            f"### \U0001f6e1\ufe0f Vulnerability Analysis\n"
            f"*   **Status**: VULNERABLE\n"
            f"*   **Type**: {vulnerability_type}\n"
            f"*   **Severity**: HIGH\n\n"
            f"### \U0001f4dd Explanation\n"
            f"{completion.strip()}\n"
        )

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": full_prompt},
        {"role": "assistant", "content": assistant_reply}
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

formatted_dataset = dataset.map(format_chat_template)
print("Sample formatted text (first 800 chars):")
print(formatted_dataset['train'][0]['text'][:800])

### Model Loading (Memory Management)
**What this cell does:** Loads the base 32B model into the GPU VRAM.

**Important Parameters:**
*   **`torch_dtype=torch.bfloat16`**: Loads the model weights natively in 16-bit precision. This maximizes the model's intelligence and takes advantage of ROCm's `bfloat16` optimizations, bypassing the need for lossy 4-bit quantization since VRAM is plentiful.

In [ ]:
# ─── Cell 6: Load Model (ROCm Optimized) ──────────────────────────────────────
if USE_4BIT:
    print(f"Loading {MODEL_SIZE} model with 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
else:
    print(f"Loading {MODEL_SIZE} model in native bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.config.use_cache = False

print("Model loaded successfully!")

### Low-Rank Adaptation (LoRA) Configuration
**What this cell does:** Sets up the LoRA adapters, allowing us to fine-tune the massive model efficiently by only training a tiny subset of extra parameters.

**Important Parameters:**
*   **`r=32` & `lora_alpha=64`**: Sets the rank (capacity) of the adapter. `r=32` provides enough complexity to learn security reasoning. `alpha` scales the adapter output, conventionally set to double the rank.
*   **`target_modules`**: Targets all attention linear layers (`q_proj`, `k_proj`, etc.) for richer adaptation.
*   **`modules_to_save=["embed_tokens", "lm_head"]` (CRITICAL)**: Forces the model's input token embeddings and output vocabulary to remain fully trainable in high precision. Without this, the model loses its Java syntax accuracy.

In [ ]:
# ─── Cell 7: LoRA Configuration ───────────────────────────────────────────────
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # modules_to_save keeps embed_tokens and lm_head fully trainable (not LoRA-reduced).
    # Without this the model loses Java syntax precision after fine-tuning.
    modules_to_save=["embed_tokens", "lm_head"]
)

### Training Arguments (SFTConfig)
**What this cell does:** Defines the hyperparameters for the Supervised Fine-Tuning (SFT) loop, prioritizing stability over speed.

**Important Parameters:**
*   **`packing=False` (CRITICAL)**: Prevents combining multiple examples into a single sequence. This avoids "cross-contamination" of examples and gradient corruption.
*   **`max_seq_length=2048`**: Ensures that long Java files are fully processed and not silently truncated mid-training.
*   **`learning_rate=1e-4`**: A very gentle learning rate. A higher rate (e.g., `2e-4`) on a 32B model causes "catastrophic forgetting" of its base coding skills.
*   **`per_device_train_batch_size=4` & `gradient_accumulation_steps=8`**: Achieves a stable effective batch size of 32. VRAM constraints limit the micro-batch size to 4 because keeping `embed_tokens` trainable uses significant memory.
*   **`warmup_steps=100`**: Gradually ramps up the learning rate to avoid an initial shock to the weights.

In [ ]:
# ─── Cell 8: SFTConfig ────────────────────────────────────────────────────────
#
# packing=False is critical:
#   packing=True concatenates examples into one long sequence. Without
#   flash_attention_2 this causes cross-contamination between samples and
#   sequences of 60,000+ tokens (model max is 32,768), corrupting gradients.
#
# max_seq_length=2048 goes in SFTConfig in TRL 1.6, NOT in SFTTrainer().
#   At 1024 (old value) complex fix explanations + code were silently
#   truncated during training — model learned to produce incomplete outputs.
#
# learning_rate=1e-4 (was 2e-4):
#   2e-4 is too aggressive for a 32B model and overwrites base Java coding
#   knowledge too fast, causing syntactic errors in the generated fix code.
#
# batch_size=4 (not 8): modules_to_save adds full-precision embed_tokens +
#   lm_head params that spike VRAM. grad_accum=8 keeps effective batch = 32.

if MODEL_SIZE == "32B" and not USE_4BIT:
    batch_size = 4
    grad_accum = 8
else:
    batch_size = 16
    grad_accum = 2

training_args = SFTConfig(
    output_dir="./large-java-vuln-model-full",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    optim="paged_adamw_32bit",
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    num_train_epochs=EPOCHS,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
)

print(f"Effective batch size : {batch_size * grad_accum}")
print(f"Packing              : False")
print(f"max_seq_length       : 2048")
print(f"learning_rate        : 1e-4")

### Initialize Trainer and Train
**What this cell does:** Starts the actual training loop using the `SFTTrainer` class, applying the dataset and configurations defined above.

In [ ]:
# ─── Cell 9: Train ────────────────────────────────────────────────────────────
# Note: max_seq_length is in SFTConfig (Cell 8), NOT here.
# Passing it here causes: TypeError: SFTTrainer.__init__() got an unexpected
# keyword argument 'max_seq_length' in TRL 1.6.

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['validation'],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.model.print_trainable_parameters()
print("Starting full training loop on 100% of the dataset...")
trainer.train()

### Save Model Artifacts
**What this cell does:** Saves the trained LoRA adapter weights and the tokenizer to disk.

**Important Note:**
*   This only saves the small adapter layers (a few hundred MBs), not the entire 32B model, making it highly storage-efficient.

In [ ]:
# ─── Cell 10: Save Adapter Weights ────────────────────────────────────────────
adapter_output_path = f"./java-vuln-adapter-{MODEL_SIZE.lower()}-full"
trainer.model.save_pretrained(adapter_output_path)
tokenizer.save_pretrained(adapter_output_path)
print(f"Training Complete! Adapter saved to '{adapter_output_path}'")